# Real World Classification

## Load data

Import the necessary libraries

In [ ]:
# If you do not use colab. You should install these packages.
# !pip install numpy
# !pip install pandas
# !pip install matplotlib
# !pip install scikit-learn
# !pip install graphviz

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

seed=40
np.random.seed(seed)

load the data

In [ ]:
# Load data from the regularization demo CSV
df = pd.read_csv('data/NYCU_Iris.csv')
df.head()

## Data Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer

def data_preprocessing(df):
    # transform label to bi-class
    df['Species'] = df['Species'].astype(str).str.strip()
    le = LabelEncoder()
    df['Species'] = le.fit_transform(df['Species'])

    feature_cols = [c for c in df.columns if c not in ['Id', 'Species']]

    # transform string to number
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # TODO: Replace the missing values using “Nearest Neighbors Imputation”
    # ---------- Start your code below ----------
    imputer = KNNImputer(n_neighbors=5)   
    df[feature_cols] = imputer.fit_transform(df[feature_cols])
    
    # ---------- --------------------- ----------
     
    return df, feature_cols

df, feature_cols = data_preprocessing(df)

In [ ]:
df.head()

In [ ]:
df.describe()

## Data Exploration

In [ ]:
from sklearn.feature_selection import r_regression

# TODO: Complete the 4. Data Exploration
plt.hist(df['PetalWidthCm'], bins=20, color='skyblue', edgecolor='black')
plt.title('Histogram of Petal Width (cm)')
plt.xlabel('Petal Width (cm)')
plt.ylabel('Frequency')
plt.show()

corr_matrix = df.corr(method='pearson')

corr_matrix = corr_matrix.drop(columns=['Id'], errors='ignore')
corr_matrix = corr_matrix.drop(index=['Id'], errors='ignore')

petal_corr = corr_matrix['PetalWidthCm'].drop(labels=['PetalWidthCm'], errors='ignore')

largest_corr_feature = petal_corr.idxmax()
largest_corr_value = petal_corr.max()

print("Feature with largest positive correlation:", largest_corr_feature)
print("Correlation coefficient:", largest_corr_value)

top5_negative_corr = petal_corr.nsmallest(5)

print("Top 5 features with the strongest negative correlations:")
print(top5_negative_corr)
features_to_plot = [largest_corr_feature] + list(top5_negative_corr.index)

plt.figure(figsize=(12, 6))
df[features_to_plot].boxplot()

plt.title("Boxplots of Selected Features")
plt.xticks(rotation=45)
plt.show()

## Model Training

### Prepare the data

In [ ]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split

# normalize the data to [0,1]
for col in feature_cols:
    col_min = df[col].min()
    col_max = df[col].max()
    if col_max > col_min:
        df[col] = (df[col] - col_min) / (col_max - col_min)
    else:
        df[col] = 0.0
        
X = df[feature_cols].values.astype(float)
y = df['Species'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)

df.head()

### Train the model!

In [ ]:
# Use 5-fold cross-validation to choose the best hyperparameters

from model.linear_model import LinearModel
from model.metrics import logloss
from model.gradients import logloss_sigmoid_grad
from model.utils import *
from model.activations import sigmoid
from sklearn.model_selection import KFold, cross_val_score

# Model configuration
loss_fn = logloss
act_fn = sigmoid
grad_fn = logloss_sigmoid_grad

learning_rates = [0.005, 0.01, 0.1, 0.5]
reg_lambdas = [1.0, 2.0, 4.0, 8.0]

# The assignment asks us to fix the cv parameter like this.
cv = KFold(n_splits=5, shuffle=True, random_state=40)

cv_results = pd.DataFrame(index=learning_rates, columns=reg_lambdas, dtype=float)
cv_results.index.name = "learning_rate"
cv_results.columns.name = "reg_lambda"

best_score = -np.inf
best_lr = None
best_reg_lambda = None

for lr in learning_rates:
    for reg_lambda in reg_lambdas:
        cv_model = LinearModel(
            dim=X_train.shape[1],
            is_reg=False,
            loss_fn=loss_fn,
            act_fn=act_fn,
            grad_fn=grad_fn,
            lr=lr,
            reg_type='l2',
            reg_lambda=reg_lambda,
            n_iteration=10000,
            val_ratio=0.2,
            random_state=seed,
            verbose=False,
            plot_curve=False
        )

        scores = cross_val_score(
            cv_model,
            X_train,
            y_train,
            cv=cv,
            scoring='accuracy'
        )

        avg_accuracy = scores.mean()
        cv_results.loc[lr, reg_lambda] = avg_accuracy

        if avg_accuracy > best_score:
            best_score = avg_accuracy
            best_lr = lr
            best_reg_lambda = reg_lambda

print("5-fold CV average accuracy table:")
display(cv_results)

print(f"Best learning rate: {best_lr}")
print(f"Best reg_lambda: {best_reg_lambda}")
print(f"Best 5-fold CV average accuracy: {best_score:.4f}")

# Train the final model on the whole training set using the best hyperparameters.
"""
np.random.seed(seed)
model = LinearModel(
    dim=X_train.shape[1],
    is_reg=False,
    loss_fn=loss_fn,
    act_fn=act_fn,
    grad_fn=grad_fn,
    lr=best_lr,
    reg_type='l2',
    reg_lambda=best_reg_lambda,
    n_iteration=10000,
    val_ratio=0.2,
    random_state=seed,
    verbose=True,
    plot_curve=True
)
model.fit(X_train, y_train)

# print model parameters
print("Model parameters (weights):", model.W)
# sum of absolute values of weights
print("Sum of absolute values of weights:", np.sum(np.abs(model.W)))
"""


In [ ]:
from model.linear_model import LinearModel
from model.metrics import logloss
from model.metrics import evaluate_binary_classifier
from model.gradients import logloss_sigmoid_grad
from model.utils import *
from model.activations import sigmoid
from sklearn.model_selection import KFold, cross_val_score
np.random.seed(seed)
model = LinearModel(
    dim=X_train.shape[1],
    is_reg=False,
    loss_fn=loss_fn,
    act_fn=act_fn,
    grad_fn=grad_fn,
    lr=0.01,
    reg_type='l2',
    reg_lambda=8.0,
    n_iteration=10000,
    val_ratio=0.2,
    random_state=seed,
    verbose=True,
    plot_curve=True
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
evaluate_binary_classifier(y_test, y_pred)

## Metrics

In [ ]:
# use evaluate_binary_classifier to evaluate the model on the test set
from model.metrics import evaluate_binary_classifier

y_pred = model.predict(X_test)
evaluate_binary_classifier(y_test, y_pred)